<a href="https://colab.research.google.com/github/oscarzgo1/AI-DEVELOPMENT-PROJECT---QSR/blob/main/Full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import time
import numpy as np

from reachy_mini import ReachyMini
from Yolo_and_Conceptnet import YOLOPipeline
from Depth_and_3D import DepthPipeline
from QSR import QSRPipeline


class FullPipeline:

    def __init__(self):

        self.collected_frames = []
        self.frame_length = 100
        self.frame_id = 0

        self.YOLOPipeline = YOLOPipeline()
        self.DepthPipeline = DepthPipeline()
        self.QSRPipeline = QSRPipeline(window_size=10)

    # Getting frames from the robot
    def get_frame(self, mini):
        while True:
            frame = mini.media.get_frame()
            if frame is not None:
                frame_display = frame.copy()
                timestamp = time.time()
                return frame, timestamp, frame_display
            time.sleep(0.05)


    def run(self): # Running loop

        with ReachyMini(
            media_backend="default",
            host="172.20.10.4",
            connection_mode="network"
        ) as mini:

            time.sleep(3)  # allow camera to start

            for _ in range(self.frame_length):

                self.frame_id += 1

                # -------------------------------
                # GET FRAME
                # -------------------------------
                frame, frame_timestamp, frame_display = self.get_frame(mini)

                # -------------------------------
                # YOLO DETECTION
                # -------------------------------
                detection_results = self.YOLOPipeline.runYolo(frame)
                detections_in_frame = self.YOLOPipeline.processDetections(
                    detection_results, frame
                )

                frame_display = self.YOLOPipeline.drawDetections(
                    frame_display, detections_in_frame
                )

                # -------------------------------
                # REQUIRE AT LEAST 2 OBJECTS
                # -------------------------------
                if len(detections_in_frame) >= 2:

                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                    # -------------------------------
                    # DEPTH + 3D PROCESSING
                    # -------------------------------
                    frame_rgb, depth, scene_package = self.DepthPipeline.process_image(
                        frame_rgb,
                        detections_in_frame,
                        self.frame_id,
                        frame_timestamp
                    )


                    depth_vis = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)
                    depth_vis = depth_vis.astype(np.uint8)
                    depth_vis = cv2.applyColorMap(depth_vis, cv2.COLORMAP_MAGMA)
                    cv2.imshow("Depth Image", depth_vis)


                    # Fixing object format
                    scene_package = self._ensure_object_ids(scene_package)

                    # store frame
                    self.collected_frames.append(scene_package)


                    self.QSRPipeline.process_frames(self.collected_frames)


                cv2.imshow("Reachy Camera", frame_display)
                cv2.waitKey(1)

        cv2.destroyAllWindows()

    # -------------------------------
    # ENSURE UNIQUE OBJECT IDS
    # -------------------------------
    def _ensure_object_ids(self, scene_package):
        """
        Converts labels → unique IDs for each object in the scene.
        """

        counts = {}

        for obj in scene_package["objects"]:
            label = obj["label"]

            if label not in counts:
                counts[label] = 0

            counts[label] += 1

            # create unique ID like person_1, person_2
            obj["id"] = f"{label}_{counts[label]}"

        return scene_package



if __name__ == "__main__": # Pipeline run
    pipeline = FullPipeline()
    pipeline.run()